<a href="https://colab.research.google.com/github/Shervinrtd/Air-Quality/blob/main/Air-Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Air Quality Prediction Using PySpark and Machine Learning
## Distributed Time-Series Forecasting for Environmental Pollution Analysis

###  Project Description

This project presents a scalable machine learning pipeline developed using **Apache Spark (PySpark)** for predicting Carbon Monoxide (`CO_GT`) concentration from environmental sensor data and temporal features.

The notebook demonstrates the complete workflow of a real-world Big Data and Machine Learning project, including:

- Distributed data preprocessing using Spark
- Sensor anomaly detection and missing-value handling
- Time-series feature engineering
- Cyclical temporal encoding
- Lag-feature generation
- Chronological train/test evaluation
- Hyperparameter tuning
- Machine learning model comparison using Spark MLlib

Two regression models were implemented and evaluated:

- **Linear Regression**
- **Random Forest Regression**

The project combines Big Data technologies and machine learning techniques to perform short-term air quality prediction and environmental analytics on time-series sensor data.

---


In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AirQualityPrediction") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


**Download CSV from GitHub**

In [3]:
!wget https://raw.githubusercontent.com/Shervinrtd/Air-Quality/main/dataset/AirQualityUCI.csv

--2026-05-21 18:01:27--  https://raw.githubusercontent.com/Shervinrtd/Air-Quality/main/dataset/AirQualityUCI.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 775593 (757K) [text/plain]
Saving to: ‘AirQualityUCI.csv.6’

AirQualityUCI.csv.6 100%[===================>] 757.42K  --.-KB/s    in 0.005s  

2026-05-21 18:01:27 (137 MB/s) - ‘AirQualityUCI.csv.6’ saved [775593/775593]



In [4]:
df = spark.read.csv(
    "AirQualityUCI.csv",
    header=True,
    sep=";"
)

df.show(5)
df.printSchema()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|
|10/03/2004|21.00.00|   2,2|       1376|      80|     9,2|          948|    

**Remove Garbage Columns**

In [5]:
df = df.drop("_c15", "_c16")

print(len(df.columns))
print(df.columns)

15
['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


**Rename Columns**

In [6]:
numeric_columns = [
    "CO_GT",
    "PT08_S1_CO",
    "NMHC_GT",
    "C6H6_GT",
    "PT08_S2_NMHC",
    "NOx_GT",
    "PT08_S3_NOx",
    "NO2_GT",
    "PT08_S4_NO2",
    "PT08_S5_O3",
    "Temperature",
    "RH",
    "AH"
]

**Replace Decimal Commas**

In [7]:
from pyspark.sql.functions import regexp_replace, col

# Get the current column names that need renaming
# These are the columns from the original DataFrame, excluding 'Date' and 'Time'
original_numeric_like_cols = [c for c in df.columns if c not in ["Date", "Time"]]

# Rename columns in df from original names to the clean names in numeric_columns
# The numeric_columns list is available from the previously executed cell WTVm94_ANbKf
for old_name, new_name in zip(original_numeric_like_cols, numeric_columns):
    df = df.withColumnRenamed(old_name, new_name)

# Now, apply regexp_replace to the newly renamed columns
for column_name in numeric_columns:
    df = df.withColumn(
        column_name,
        regexp_replace(col(column_name), ",", ".")
    )

In [8]:
from pyspark.sql.types import DoubleType

for column in numeric_columns:
    df = df.withColumn(
        column,
        col(column).cast(DoubleType())
    )

df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO_GT: double (nullable = true)
 |-- PT08_S1_CO: double (nullable = true)
 |-- NMHC_GT: double (nullable = true)
 |-- C6H6_GT: double (nullable = true)
 |-- PT08_S2_NMHC: double (nullable = true)
 |-- NOx_GT: double (nullable = true)
 |-- PT08_S3_NOx: double (nullable = true)
 |-- NO2_GT: double (nullable = true)
 |-- PT08_S4_NO2: double (nullable = true)
 |-- PT08_S5_O3: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



In [9]:
from pyspark.sql.functions import when

for column in numeric_columns:
    df = df.withColumn(
        column,
        when(col(column) == -200.0, None)
        .otherwise(col(column))
    )

In [10]:
df = df.filter(col("CO_GT").isNotNull())

print("Rows after removing missing target:", df.count())

Rows after removing missing target: 7674


**Convert Numeric Columns**

**Remove Missing Target Rows**

**Drop Weak Feature**

In [11]:
df = df.drop("NMHC_GT")

df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO_GT: double (nullable = true)
 |-- PT08_S1_CO: double (nullable = true)
 |-- C6H6_GT: double (nullable = true)
 |-- PT08_S2_NMHC: double (nullable = true)
 |-- NOx_GT: double (nullable = true)
 |-- PT08_S3_NOx: double (nullable = true)
 |-- NO2_GT: double (nullable = true)
 |-- PT08_S4_NO2: double (nullable = true)
 |-- PT08_S5_O3: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



**Create Time Feature**

In [12]:
from pyspark.sql.functions import concat_ws, to_timestamp

# Combine Date and Time into timestamp

df = df.withColumn(
    "Datetime",
    to_timestamp(
        concat_ws(" ", col("Date"), col("Time")),
        "dd/MM/yyyy HH.mm.ss"
    )
)

df.select("Date", "Time", "Datetime").show(5)

+----------+--------+-------------------+
|      Date|    Time|           Datetime|
+----------+--------+-------------------+
|10/03/2004|18.00.00|2004-03-10 18:00:00|
|10/03/2004|19.00.00|2004-03-10 19:00:00|
|10/03/2004|20.00.00|2004-03-10 20:00:00|
|10/03/2004|21.00.00|2004-03-10 21:00:00|
|10/03/2004|22.00.00|2004-03-10 22:00:00|
+----------+--------+-------------------+
only showing top 5 rows


In [13]:
from pyspark.sql.functions import hour, month, dayofweek

# Extract time-based features

df = df.withColumn("Hour", hour(col("Datetime")))
df = df.withColumn("Month", month(col("Datetime")))
df = df.withColumn("DayOfWeek", dayofweek(col("Datetime")))

df.select(
    "Datetime",
    "Hour",
    "Month",
    "DayOfWeek"
).show(5)

+-------------------+----+-----+---------+
|           Datetime|Hour|Month|DayOfWeek|
+-------------------+----+-----+---------+
|2004-03-10 18:00:00|  18|    3|        4|
|2004-03-10 19:00:00|  19|    3|        4|
|2004-03-10 20:00:00|  20|    3|        4|
|2004-03-10 21:00:00|  21|    3|        4|
|2004-03-10 22:00:00|  22|    3|        4|
+-------------------+----+-----+---------+
only showing top 5 rows


In [14]:
import math
from pyspark.sql.functions import sin, cos, col

# Cyclical encoding for Hour
df = df.withColumn(
    "Hour_Sin",
    sin(col("Hour") * (2 * math.pi / 24))
)

df = df.withColumn(
    "Hour_Cos",
    cos(col("Hour") * (2 * math.pi / 24))
)

# Cyclical encoding for Month
df = df.withColumn(
    "Month_Sin",
    sin(col("Month") * (2 * math.pi / 12))
)

df = df.withColumn(
    "Month_Cos",
    cos(col("Month") * (2 * math.pi / 12))
)

# Cyclical encoding for DayOfWeek
df = df.withColumn(
    "DayOfWeek_Sin",
    sin(col("DayOfWeek") * (2 * math.pi / 7))
)

df = df.withColumn(
    "DayOfWeek_Cos",
    cos(col("DayOfWeek") * (2 * math.pi / 7))
)

df.select(
    "Hour",
    "Hour_Sin",
    "Hour_Cos"
).show(5)

+----+-------------------+--------------------+
|Hour|           Hour_Sin|            Hour_Cos|
+----+-------------------+--------------------+
|  18|               -1.0|-1.83697019872102...|
|  19|-0.9659258262890684|  0.2588190451025203|
|  20| -0.866025403784439| 0.49999999999999933|
|  21|-0.7071067811865477|  0.7071067811865474|
|  22|-0.5000000000000004|  0.8660254037844384|
+----+-------------------+--------------------+
only showing top 5 rows


In [15]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

# Remove null timestamps
df = df.filter(col("Datetime").isNotNull())

# Sort chronologically
df = df.orderBy("Datetime")

# Create time window
time_window = Window.orderBy("Datetime")

# Create lag features
df = df.withColumn(
    "CO_GT_Lag1",
    lag("CO_GT", 1).over(time_window)
)

df = df.withColumn(
    "CO_GT_Lag2",
    lag("CO_GT", 2).over(time_window)
)

# Remove rows with null lag values
df = df.dropna(
    subset=["CO_GT_Lag1", "CO_GT_Lag2"]
)

df.select(
    "Datetime",
    "CO_GT",
    "CO_GT_Lag1",
    "CO_GT_Lag2"
).show(10)

+-------------------+-----+----------+----------+
|           Datetime|CO_GT|CO_GT_Lag1|CO_GT_Lag2|
+-------------------+-----+----------+----------+
|2004-03-10 20:00:00|  2.2|       2.0|       2.6|
|2004-03-10 21:00:00|  2.2|       2.2|       2.0|
|2004-03-10 22:00:00|  1.6|       2.2|       2.2|
|2004-03-10 23:00:00|  1.2|       1.6|       2.2|
|2004-03-11 00:00:00|  1.2|       1.2|       1.6|
|2004-03-11 01:00:00|  1.0|       1.2|       1.2|
|2004-03-11 02:00:00|  0.9|       1.0|       1.2|
|2004-03-11 03:00:00|  0.6|       0.9|       1.0|
|2004-03-11 05:00:00|  0.7|       0.6|       0.9|
|2004-03-11 06:00:00|  0.7|       0.7|       0.6|
+-------------------+-----+----------+----------+
only showing top 10 rows


**Missing Value Analysis**

In [16]:
from pyspark.sql.functions import isnan, when, count

missing_values = df.select([
    count(
        when(
            col(c).isNull() | isnan(c),
            c
        )
    ).alias(c)
    for c in df.columns
    if c not in ["Date", "Time", "Datetime"]
])

missing_values.show()

+-----+----------+-------+------------+------+-----------+------+-----------+----------+-----------+---+---+----+-----+---------+--------+--------+---------+---------+-------------+-------------+----------+----------+
|CO_GT|PT08_S1_CO|C6H6_GT|PT08_S2_NMHC|NOx_GT|PT08_S3_NOx|NO2_GT|PT08_S4_NO2|PT08_S5_O3|Temperature| RH| AH|Hour|Month|DayOfWeek|Hour_Sin|Hour_Cos|Month_Sin|Month_Cos|DayOfWeek_Sin|DayOfWeek_Cos|CO_GT_Lag1|CO_GT_Lag2|
+-----+----------+-------+------------+------+-----------+------+-----------+----------+-----------+---+---+----+-----+---------+--------+--------+---------+---------+-------------+-------------+----------+----------+
|    0|       330|    330|         330|   413|        330|   416|        330|       330|        330|330|330|   0|    0|        0|       0|       0|        0|        0|            0|            0|         0|         0|
+-----+----------+-------+------------+------+-----------+------+-----------+----------+-----------+---+---+----+-----+---------

**Correlation Analysis**

In [17]:
correlation_columns = [
    "PT08_S1_CO",
    "C6H6_GT",
    "PT08_S2_NMHC",
    "NOx_GT",
    "PT08_S3_NOx",
    "NO2_GT",
    "PT08_S4_NO2",
    "PT08_S5_O3",
    "Temperature",
    "RH",
    "AH",
    "Hour",
    "Month",
    "DayOfWeek"
]

for column in correlation_columns:
    correlation = df.stat.corr("CO_GT", column)
    print(f"Correlation between CO_GT and {column}: {correlation}")

Correlation between CO_GT and PT08_S1_CO: 0.5312227955197484
Correlation between CO_GT and C6H6_GT: 0.8449315883365438
Correlation between CO_GT and PT08_S2_NMHC: 0.6671477889907094
Correlation between CO_GT and NOx_GT: 0.7886035260944007
Correlation between CO_GT and PT08_S3_NOx: -0.6072360392724383
Correlation between CO_GT and NO2_GT: 0.6569826783140007
Correlation between CO_GT and PT08_S4_NO2: 0.41487541748025863
Correlation between CO_GT and PT08_S5_O3: 0.6935007159742642
Correlation between CO_GT and Temperature: -0.008966803896614708
Correlation between CO_GT and RH: 0.003168589398868354
Correlation between CO_GT and AH: 0.007639302926354461
Correlation between CO_GT and Hour: 0.358184816811422
Correlation between CO_GT and Month: 0.11124635348842092
Correlation between CO_GT and DayOfWeek: 0.11748130542358794


In [18]:
feature_columns = [
    "PT08_S1_CO",
    "C6H6_GT",
    "PT08_S2_NMHC",
    "NOx_GT",
    "PT08_S3_NOx",
    "NO2_GT",
    "PT08_S4_NO2",
    "PT08_S5_O3",
    "Temperature",
    "RH",
    "AH",

    # Cyclical time features
    "Hour_Sin",
    "Hour_Cos",
    "Month_Sin",
    "Month_Cos",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos",

    # Lag features
    "CO_GT_Lag1",
    "CO_GT_Lag2"
]

**Train/Test Split**

In [19]:
from pyspark.sql.functions import col

# Use timestamp-based chronological split

split_date = "2004-11-01 00:00:00"

train_data = df.filter(
    col("Datetime") < split_date
)

test_data = df.filter(
    col("Datetime") >= split_date
)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 4205
Testing rows: 3467


In [20]:
from pyspark.ml.feature import Imputer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Imputer
imputer = Imputer(
    inputCols=feature_columns,
    outputCols=feature_columns
).setStrategy("mean")

# Vector Assembler
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

# Feature Scaling
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

# Linear Regression
lr = LinearRegression(
    featuresCol="scaled_features",
    labelCol="CO_GT"
)

# Create Pipeline
pipeline = Pipeline(stages=[
    imputer,
    assembler,
    scaler,
    lr
])

In [21]:
pipeline_model = pipeline.fit(train_data)

In [22]:
lr_predictions = pipeline_model.transform(test_data)

lr_predictions.select(
    "CO_GT",
    "prediction"
).show(10)

+-----+------------------+
|CO_GT|        prediction|
+-----+------------------+
|  3.2| 2.396957243273316|
|  3.7|2.6618280138856103|
|  3.5|2.7727446684022627|
|  3.0| 2.538809953872065|
|  2.7|2.2765290757297016|
|  1.8| 1.465114823725672|
|  1.3|1.1214036918043802|
|  1.5|1.2744573521185354|
|  1.9|1.5305346338790091|
|  2.2| 1.816573382511375|
+-----+------------------+
only showing top 10 rows


In [23]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE evaluator

evaluator_rmse = RegressionEvaluator(
    labelCol="CO_GT",
    predictionCol="prediction",
    metricName="rmse"
)

# R2 evaluator

evaluator_r2 = RegressionEvaluator(
    labelCol="CO_GT",
    predictionCol="prediction",
    metricName="r2"
)

rmse_lr = evaluator_rmse.evaluate(lr_predictions)
r2_lr = evaluator_r2.evaluate(lr_predictions)

print("Linear Regression RMSE:", rmse_lr)
print("Linear Regression R2:", r2_lr)

Linear Regression RMSE: 0.569888314324246
Linear Regression R2: 0.8731204692067995


In [33]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Random Forest model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="CO_GT",
    seed=42
)

# Random Forest pipeline
rf_pipeline = Pipeline(stages=[
    imputer,
    assembler,
    rf
])

# Chronological validation split INSIDE training data
tune_split_date = "2004-09-01 00:00:00"

tune_train = train_data.filter(
    col("Datetime") < tune_split_date
)

tune_val = train_data.filter(
    col("Datetime") >= tune_split_date
)

# Cache datasets for faster tuning
tune_train = tune_train.cache()
tune_val = tune_val.cache()

print("Tune Train Rows:", tune_train.count())
print("Tune Validation Rows:", tune_val.count())

# Hyperparameter grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10, 15])
    .addGrid(rf.numTrees, [50, 100])
    .build()
)

# Manual chronological tuning loop
best_rmse = float("inf")
best_params = None

print("Starting chronological hyperparameter tuning...")

for params in paramGrid:

    # Train model with parameters
    model = rf_pipeline.fit(tune_train, params)

    # Predict on future validation set
    predictions = model.transform(tune_val)

    # Evaluate RMSE
    rmse = evaluator_rmse.evaluate(predictions)

    # Extract tested parameters
    depth = params[rf.maxDepth]
    trees = params[rf.numTrees]

    print(
        f"Tested depth={depth}, "
        f"trees={trees} "
        f"-> Validation RMSE: {rmse:.4f}"
    )

    # Save best parameters
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params

print("\nBest Parameters Found!")

# Retrain on FULL training data using best parameters
final_rf_model = rf_pipeline.fit(
    train_data,
    best_params
)

# Final predictions on untouched test set
final_predictions = final_rf_model.transform(test_data)

# Final evaluation
rmse_rf_tuned = evaluator_rmse.evaluate(final_predictions)
r2_rf_tuned = evaluator_r2.evaluate(final_predictions)

print("\n=== FINAL TUNED MODEL SCORES ===")
print("Tuned Random Forest RMSE:", rmse_rf_tuned)
print("Tuned Random Forest R2:", r2_rf_tuned)

Tune Train Rows: 3256
Tune Validation Rows: 949
Starting chronological hyperparameter tuning...
Tested depth=5, trees=50 -> Validation RMSE: 0.4287
Tested depth=5, trees=100 -> Validation RMSE: 0.4426
Tested depth=10, trees=50 -> Validation RMSE: 0.4090
Tested depth=10, trees=100 -> Validation RMSE: 0.4232
Tested depth=15, trees=50 -> Validation RMSE: 0.4088
Tested depth=15, trees=100 -> Validation RMSE: 0.4240

Best Parameters Found!

=== FINAL TUNED MODEL SCORES ===
Tuned Random Forest RMSE: 0.6627655217112384
Tuned Random Forest R2: 0.8283942342796384
